# Advanced Analysis — Bike Sharing Dataset

Notebook tambahan ini dibuat untuk melengkapi submission **Proyek Analisis Data** dengan teknik analisis lanjutan tanpa machine learning.

## Tujuan Analisis
Menerapkan **clustering manual berbasis binning** untuk mengelompokkan tingkat permintaan penyewaan sepeda menjadi tiga segmen: **Low Demand, Medium Demand, dan High Demand**. Segmentasi ini membantu mengidentifikasi kapan kebutuhan ketersediaan sepeda paling tinggi sehingga dapat digunakan sebagai dasar keputusan operasional.


## Pertanyaan Analisis Lanjutan
1. Bagaimana distribusi jam operasional pada setiap segmen permintaan selama periode 2011–2012?
2. Musim dan tipe hari apa yang paling sering masuk ke segmen **High Demand**, sehingga dapat menjadi prioritas penambahan ketersediaan sepeda?


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


## 1. Gathering Data
Dataset yang digunakan adalah `hour.csv` dari Bike Sharing Dataset yang telah disediakan Dicoding. Data berisi jumlah penyewaan per jam beserta informasi tanggal, musim, cuaca, hari kerja, dan karakteristik waktu lainnya.


In [2]:
df = pd.read_csv('data/hour.csv')
df['dteday'] = pd.to_datetime(df['dteday'])
df.head()


### Insight Gathering
Kolom `cnt` digunakan sebagai ukuran total permintaan penyewaan sepeda per jam. Kolom `hr`, `season`, `workingday`, dan `dteday` digunakan untuk membaca pola waktu dan kondisi operasional.


## 2. Assessing Data
Sebelum melakukan segmentasi, kualitas data diperiksa kembali melalui missing value, duplikasi, tipe data, dan konsistensi nilai target `cnt`.


In [3]:
print('Missing value:')
print(df.isna().sum())
print('\nJumlah duplikasi:', df.duplicated().sum())
print('\nRentang cnt:', df['cnt'].min(), '-', df['cnt'].max())


### Insight Assessing
Pemeriksaan dilakukan untuk memastikan tidak ada masalah kualitas data yang dapat mengganggu pembentukan segmen. Jika ditemukan nilai kosong atau duplikat, data tersebut perlu ditangani sebelum analisis. Nilai `cnt` juga diperiksa agar tidak terdapat nilai negatif yang tidak masuk akal untuk jumlah penyewaan.


## 3. Cleaning Data
Data dibersihkan secara defensif dengan menghapus duplikasi, menghapus baris yang tidak memiliki nilai `cnt`, serta memastikan `cnt` tidak negatif. Langkah ini menjaga analisis tetap dapat direproduksi meskipun file data mengalami perubahan kecil.


In [4]:
clean_df = df.drop_duplicates().dropna(subset=['cnt']).copy()
clean_df = clean_df[clean_df['cnt'] >= 0].copy()
print('Jumlah baris setelah cleaning:', len(clean_df))


### Insight Cleaning
Dataset yang sudah dibersihkan digunakan sebagai dasar seluruh proses segmentasi agar setiap kategori permintaan berasal dari observasi yang valid.


## 4. Advanced Analysis — Manual Clustering dengan Binning
Teknik yang digunakan adalah **binning berbasis kuantil** dengan `pandas.qcut`. Ini bukan algoritma machine learning. Setiap observasi dibagi menjadi tiga kelompok berdasarkan distribusi aktual `cnt`:
- **Low Demand**: sepertiga observasi dengan permintaan terendah.
- **Medium Demand**: sepertiga observasi di tengah.
- **High Demand**: sepertiga observasi dengan permintaan tertinggi.

Pendekatan berbasis kuantil dipilih agar pembagian kelompok mengikuti karakteristik distribusi data dan tidak menggunakan batas yang arbitrer.


In [5]:
labels = ['Low Demand', 'Medium Demand', 'High Demand']
clean_df['demand_segment'] = pd.qcut(clean_df['cnt'], q=3, labels=labels, duplicates='drop')
segment_summary = clean_df.groupby('demand_segment', observed=False)['cnt'].agg(['count', 'min', 'median', 'mean', 'max']).round(2)
segment_summary


### Interpretasi Segmentasi
Tabel di atas menunjukkan karakteristik tiap segmen. Segmen **High Demand** merepresentasikan periode yang relatif paling padat dibandingkan keseluruhan distribusi data, sehingga menjadi fokus utama dalam perencanaan kapasitas.


## 5. Pola Jam pada Setiap Segmen Permintaan
Analisis berikut menghitung jumlah observasi pada setiap jam untuk masing-masing segmen. Tujuannya adalah menemukan jam yang paling sering berada pada kondisi permintaan tinggi.


In [6]:
hour_segment = (clean_df.groupby(['hr', 'demand_segment'], observed=False)
                .size()
                .reset_index(name='frequency'))

plt.figure(figsize=(12, 6))
sns.lineplot(data=hour_segment, x='hr', y='frequency', hue='demand_segment', marker='o')
plt.title('Frekuensi Segmen Permintaan Berdasarkan Jam')
plt.xlabel('Jam')
plt.ylabel('Jumlah Observasi')
plt.xticks(range(24))
plt.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()


In [7]:
high_by_hour = (clean_df[clean_df['demand_segment'] == 'High Demand']
                .groupby('hr')['cnt']
                .agg(['count', 'mean'])
                .sort_values(['count', 'mean'], ascending=False))
high_by_hour.head(10)


### Insight Jam Operasional
Jam dengan frekuensi **High Demand** terbesar merupakan kandidat waktu prioritas untuk redistribusi sepeda, pengecekan ketersediaan dock, dan penambahan kesiapan operasional. Dengan cara ini, keputusan tidak hanya didasarkan pada rata-rata `cnt`, tetapi juga pada seberapa konsisten jam tersebut muncul sebagai periode permintaan tinggi.


## 6. High Demand Berdasarkan Musim dan Hari Kerja
Tahap ini melihat proporsi observasi **High Demand** pada kombinasi musim dan status hari kerja. Proporsi digunakan agar perbandingan tidak bias hanya karena jumlah observasi tiap kelompok berbeda.


In [8]:
season_map = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
working_map = {0: 'Non-Working Day', 1: 'Working Day'}
clean_df['season_name'] = clean_df['season'].map(season_map)
clean_df['workingday_name'] = clean_df['workingday'].map(working_map)

group_total = clean_df.groupby(['season_name', 'workingday_name']).size().rename('total_observation')
group_high = (clean_df[clean_df['demand_segment'] == 'High Demand']
              .groupby(['season_name', 'workingday_name'])
              .size()
              .rename('high_demand_observation'))
high_share = pd.concat([group_total, group_high], axis=1).fillna(0)
high_share['high_demand_share_pct'] = (high_share['high_demand_observation'] / high_share['total_observation'] * 100).round(2)
high_share.sort_values('high_demand_share_pct', ascending=False)


In [9]:
plot_df = high_share.reset_index()
plt.figure(figsize=(10, 6))
sns.barplot(data=plot_df, x='season_name', y='high_demand_share_pct', hue='workingday_name')
plt.title('Proporsi High Demand Berdasarkan Musim dan Tipe Hari')
plt.xlabel('Musim')
plt.ylabel('High Demand (%)')
plt.tight_layout()
plt.show()


### Insight Musim dan Hari Kerja
Kombinasi musim dan tipe hari dengan persentase **High Demand** paling besar dapat diprioritaskan dalam perencanaan kapasitas. Penggunaan proporsi membuat rekomendasi lebih adil karena memperhitungkan jumlah observasi pada masing-masing kombinasi kondisi.


## 7. Conclusion & Recommendation
### Kesimpulan
1. Segmentasi berbasis kuantil berhasil membagi permintaan penyewaan ke dalam tiga kelompok operasional: Low, Medium, dan High Demand. Segmen High Demand dapat digunakan sebagai indikator periode yang membutuhkan perhatian kapasitas lebih tinggi.
2. Distribusi High Demand tidak merata sepanjang hari. Jam-jam yang paling sering masuk kategori High Demand menjadi kandidat utama untuk perencanaan redistribusi sepeda sebelum lonjakan permintaan terjadi.
3. Proporsi High Demand juga berbeda berdasarkan musim dan status hari kerja, sehingga strategi kapasitas sebaiknya tidak menggunakan satu aturan yang sama untuk semua kondisi.

### Action Items
- Jadwalkan **redistribusi sepeda sebelum jam-jam dengan frekuensi High Demand tertinggi**.
- Tambahkan monitoring ketersediaan sepeda dan dock pada kombinasi musim/hari dengan **proporsi High Demand terbesar**.
- Gunakan segmen Low–Medium–High sebagai indikator sederhana pada dashboard operasional sehingga tim dapat membaca tingkat permintaan tanpa membutuhkan model machine learning.
